# SRAM vs DRAM vs Flash

## 1) Bức tranh lớn: SRAM, DRAM và Flash khác nhau ở đâu?

Ba loại này đều là “memory”, nhưng chúng phục vụ vai trò rất khác nhau:
- **SRAM**: rất nhanh, rất đắt, dung lượng nhỏ, mất điện là mất dữ liệu.
- **DRAM**: dung lượng lớn hơn SRAM, rẻ hơn SRAM, chậm hơn SRAM, mất điện là mất dữ liệu.
- **Flash**: chậm hơn RAM nhiều, nhưng mất điện vẫn giữ dữ liệu.

Điểm mấu chốt không chỉ là “nhanh hay chậm”, mà là:

1. Cách lưu 1 bit dữ liệu

2. Có cần refresh không

3. Tốc độ đọc/ghi

4. Năng lượng tiêu thụ

5. Chi phí trên mỗi bit

6. Dữ liệu có còn sau khi tắt nguồn không

7. Phù hợp với vai trò nào trong hệ thống AI embeded

## 2) Bản chất vật lý của từng loại

### **2.1 SRAM là gì?**

SRAM = **Static RAM**

Một bit trong SRAM thường được giữ bằng một mạch dạng flip-flop từ nhiều transistor. Nghĩa là bit 0/1 được giữ ở trạng thái ổn định của mạch.

Ý nghĩa thực tế:

- Không cần “nạp lại” dữ liệu định kỳ
- Truy cập rất nhanh
- Độ trễ rất thấp
- Nhưng vì mỗi bit tốn nhiều transistor hơn, nên cực kỳ tốn diện tích silicon

Kết quả:

- Nhanh
- Đắt
- Ít dung lượng

SRAM thường nằm ở nơi CPU cần truy cập liên tục với tốc độ cao nhất:
- CPU cache L1/L2/L3
- on-chip buffer
- tensor arena trong MCU TinyML
- scartchpad memory cho DSP/NPU

### **2.2 DRAM là gì?**

DRAM = **Dynamic RAM**

Mỗi bit trong DRAM thường được lưu bằng:
- 1 transistor
- 1 tụ điện

Tụ điện sẽ bị rò điện theo thời gian. Vì thế nếu không làm gì, bit sẽ dần mất. Nên DRAM phải được **refresh định kỳ** để nạp lại điện tích.

Ý nghĩa thực tế:
- Cấu trúc 1 bit gọn hơn SRAM
- Nhét được nhiều bit hơn trên cùng diện tích chip
- Rẻ hơn SRAM rất nhiều
- Nhưng chậm hơn SRAM và cần logic refresh

Kết quả:
- Dung lượng lớn
- Rẻ hơn SRAM
- Chậm hơn SRAM
- Vẫn là volatile memory: mất điện là mất dữ liệu

DRAM thường là:
- RAM chính của PC
- LPDDR trên điện thoại
- DDR/LPDDR trong edge device chạy Linux như Jetson, Raspberry Pi, RK3588, NPU SoC

### **2.3 Flash là gì?**

Flash là bộ nhớ **non-volatile**.

Nó giữ dữ liệu nhờ điện tích bị “nhốt” trong transistor cổng nổi hoặc charge-trap. Vì điện tích này được giữ lại khá lâu, nên khi tắt nguồn dữ liệu vẫn còn.

Ý nghĩa thực tế:
- Giữ firmware
- Giữ model
- Giữ file, ảnh, logs, config
- Nhưng thao tác ghi/xóa chậm và có giới hạn số chu kỳ

Kết quả:
- Không mất dữ liệu khi mất điện
- Rất phù hợp cho lưu trữ
- Không phù hợp để làm vùng tính toán nhanh như RAM

Flash gồm hai nhóm hay gặp:
- **NOR Flash**:
    - Đọc ngẫu nhiên tốt hơn
    - Có thể hỗ trợ XIP (execute in place), tức CPU đọc code trực tiếp từ flash
    - Thường dùng trong MCU để chứa firmware/model nhỏ
- **NAND Flash**:
    - Mật độ lưu trữ cao hơn, rẻ hơn theo GB
    - Thường dùng trong SSD, eMMC, UFS, thẻ nhớ
    - Không lý tưởng để CPU random access như NOR

## 3) So sánh trực tiếp theo góc nhìn hệ thống

### 3.1 Volatile vs non-volatile

- SRAM: volatile
- DRAM: volatile
- Flash: non-volatile

Nghĩa là, bạn tắt nguồn MCU đi:
- SRAM mất
- DRAM mất
- Flash còn

Đây là lý do:
- chương trình và model thường được lưu trong Flash
- dữ liệu đang chạy và intermediate tensors nằm trong SRAM/DRAM

### 3.2 Tốc độ

Thứ tự thường là:
$$
SRAM > DRAM > Flash
$$

Nhưng không chỉ “nhanh/chậm” đơn giản. Nó còn là:
- **Latency**: mất bao lâu để lấy được dữ liệu đầu tiên
- **Bandwidth**: mỗi giây đọc được bao nhiêu dữ liệu
- **Random access**: truy cập lẻ tẻ có nhanh không
- **Write behavior**: ghi có dễ không

Trong AI inference, điều này rất quan trọng vì model không chỉ cần “đọc một lần”, mà phải đọc:
- weights
- activations
- feature maps
- buffers
- lookup tables

rất nhiều lần

### 3.3 Mật độ lưu trữ và chi phí

Thứ tự thường rẻ dần là:
$$
SRAM > DRAM > Flash
$$

Nên:
- Muốn chứa firmware + model lâu dài: chọn Flash
- Muốn có RAM lớn để chạy Linux/vision model: chọn DRAM
- Muốn vùng truy cập siêu nhanh sát CPU/NPU: dùng SRAM

### 3.4 Năng lượng

Không phải cứ Flash là tiết kiệm nhất trong mọi tình huống.

Cần tách ra:
- Năng lượng khi giữ dữ liệu
- Năng lượng khi đọc/ghi
- Năng lượng toàn hệ thống

**SRAM**:
- Không cần refresh
- Truy cập nhanh
- Tốt cho xử lý tức thời
- Nhưng diện tích lớn, dung lượng ít

**DRAM**:
- Có chi phí refresh
- Khi hệ thống lớn và luôn bật, DRAM có thể tốn năng lượng đáng kể

**Flash**:
- Giữ dữ liệu không cần điện liên tục
- Nhưng ghi/xóa khá tốn và chậm
- Không hợp để làm nơi CPU ghi đọc liên tục trong vòng lặp inference

## 4) Ví dụ rất đời thường để hình dung

### 4.1 Ví dụ bàn làm việc
- SRAM = tờ giấy đang đặt ngay trước mặt trên bàn
- DRAM = ngăn kéo cạnh bàn
- Flash = tủ hồ sơ

Khi làm việc:
- thứ cần dùng liên tục để suy nghĩ: đặt trên bàn -> SRAM
- thứ cần tra thường xuyên nhưng không phải từng mili giây: bỏ ngăn kéo -> DRAM
- tài liệu lưu lâu dài: bỏ tủ -> Flash

### 4.2 Ví dụ trong app camera AI

Khi thiết bị edge AI nhận một frame từ camera:
- firmware và model nằm ở Flash
- frame hiện tại được đẩy vào SRAM hoặc DRAM
- các tensor trung gian của CNN cũng nằm ở SRAM hoặc DRAM
- kết quả detection có thể lưu tạm trong RAM
- log sự kiện, ảnh chụp, model update được ghi về Flash

## 5) Trong TinyML, SRAM quan trọng hơn nhiều người mới học tưởng

Đây là điểm cực kỳ quan trọng.

Người mới thường chỉ nhìn:
- model nặng bao nhiêu MB?
- chip có đủ Flash không?

Nhưng trong TinyML, rất nhiều khi không chết vì **Flash**, mà chết vì **SRAM**.

Khi inference, model không chỉ cần lưu weights. Nó còn cần:
- input buffer
- output buffer
- tensor trung gian
- stack
- heap
- audio window / feature window
- DMA buffers
- RTOS task memory

Nên một model có thể:
- fit vào Flash
- nhưng không fit vào SRAM

Đây là lỗi rất hay gặp.

**Ví dụ**:

Một model **int8** kích thước **250 KB** có thể lưu tốt trong **Flash 1 MB**.

Nhưng nếu peak activation cần 180 KB, cộng thêm:
- audio buffer 40 KB
- stack + app logic 30 KB
- RTOS 20 KB

thì con MCU có 256 KB SRAM có thể vẫn không chạy nổi.

## 6) TinyML thường dùng SRAM và Flash như thế nào?

### 6.1 Cấu trúc điển hình của MCU TinyML

Với một MCU như STM32, nRF, ESP32, RP2040-class, hoặc board TinyML nói chung:
- **Flash** chứa:
    - firmware
    - model .tflite hoặc mảng weights C
    - constants
    - calibration table
    - labels

- **SRAM** chứa:
    - tensor arena
    - input/output tensors
    - stack
    - heap
    - sensor buffers
    - audio ring buffer
    - intermediate activations


Đây là pattern cực kỳ phổ biến trong TensorFlow Lite Micro.

### 6.2 Trong TFLite Micro

Rất thường:
- model được nhúng thành mảng const unsigned char model[]
- model đó nằm ở Flash
- còn tensor_arena nằm ở SRAM

Ví dụ tư duy:
- weights: ở Flash
- activations/workspace: ở SRAM

Cho nên khi tối ưu TinyML:
- giảm kích thước model giúp giảm Flash
- giảm peak activation giúp giảm SRAM
- cả hai đều quan trọng, nhưng SRAM thường khó hơn

## 7) Flash trong TinyML: lưu cái gì, giới hạn gì?

### 7.1 Flash phù hợp để lưu lâu dài

Trong thiết bị TinyML, Flash thường lưu:
- bootloader
- firmware
- model
- tham số cấu hình
- threshold
- feature normalization constants
- labels
- occasionally cached data hoặc logs

Ví dụ:
- thiết bị wake-word luôn bật
- firmware và mô hình “Hey Device” nằm trong Flash
- bật nguồn lên là chạy ngay từ đó

### 7.2 Không nên dùng Flash như RAM

Flash không phù hợp để làm vùng tính toán liên tục vì:
- ghi chậm
- xóa theo block/page
- có giới hạn chu kỳ ghi/xóa
- random write rất bất tiện

Nên bạn không thể coi Flash như “RAM thêm”.

### 7.3 Wear-out

Mỗi ô flash chỉ ghi/xóa được hữu hạn số lần.

Nên nếu bạn thiết kế thiết bị edge:
- ghi log liên tục mỗi 10 ms vào flash
- hoặc cập nhật model liên tục thì sẽ nhanh hao mòn.

Ví dụ sai:
- mỗi lần model ra xác suất là ghi cả vector probability vào Flash

Ví dụ hợp lý:
- chỉ ghi sự kiện quan trọng
- hoặc dùng wear leveling
- hoặc gom batch rồi mới ghi

## 8) DRAM trong Edge AI khác TinyML ở chỗ nào?

### 8.1 TinyML truyền thống thường không có DRAM

Nhiều MCU TinyML không dùng DRAM riêng như máy tính hay điện thoại. Chúng chủ yếu có:
- on-chip SRAM
- on-chip Flash
- đôi khi external QSPI flash
- đôi khi pseudo-SRAM/PSRAM

Nên bài toán tối ưu rất khắc nghiệt.

### 8.2 Edge AI chạy Linux thường dựa rất nhiều vào DRAM

Với những thiết bị như:
- Raspberry Pi
- Jetson Nano / Orin
- RK3588
- Qualcomm edge SoC
- NXP i.MX
- Intel NUC
- Android device

thì DRAM là vùng làm việc chính:
- OS
- app
- model runtime
- frame buffers
- tensor buffers
- video pipeline
- NPU shared memory

### 8.3 On-chip SRAM vẫn cực kỳ quan trọng trong Edge AI

Dù hệ thống có DRAM, accelerator vẫn thường có:
- local SRAM
- cache
- scratchpad

Đây là nơi giữ:
- tile feature maps
- tile weights
- DMA staging buffers

Vì đọc DRAM liên tục sẽ chậm và tốn điện hơn so với dùng SRAM gần compute unit.

## 9) Điểm khác biệt quyết định trong AI inference

### 9.1 Weights vs activations

Trong AI:
- **weights** là tham số model
- **activations** là dữ liệu trung gian khi tính toán

Từ góc nhìn bộ nhớ:
- weights thường “lưu lâu dài” -> phù hợp Flash hoặc DRAM
- activations cần đọc/ghi liên tục -> phải ở SRAM hoặc DRAM

Trong TinyML:
- weights hay để Flash
- activations để SRAM

Trong edge device lớn:
- weights thường nằm DRAM khi runtime
- activations ở DRAM, cache SRAM cục bộ, hoặc SRAM của accelerator

### 9.2 Vì sao SRAM thường là nút thắt cổ chai?

Ví dụ một conv layer trên ảnh:
- input feature map 96x96x32
- output feature map 48x48x64
- cần buffer cho im2col/tiled ops
- cần workspace cho kernel

Chỉ riêng activation có thể vượt xa kích thước weights ở một số layer.

Nên có những model:
- weights nhỏ nhờ quantization
- nhưng activation vẫn lớn

Điều này đặc biệt hay xảy ra ở:
- early CNN layers với ảnh lớn
- models xử lý audio spectrogram dài
- segmentation
- object detection có nhiều head

## 10) Ứng dụng thực tế trong TinyML

### 10.1 Wake word / openWakeWord / keyword spotting

Đây là use case rất hợp với TinyML.

Pipeline điển hình:
1. Microphone lấy audio stream
2. Audio samples lưu vào SRAM
3. Tạo feature như MFCC/mel spectrogram trong SRAM
4. Model weights nằm trong Flash
5. Inference dùng:
- weights từ Flash
- activations trong SRAM
6. Nếu phát hiện wake word:
- ghi event vào Flash
- hoặc gửi lên cloud


Ở đây:
- SRAM là nơi cực kỳ quan trọng vì:
    - audio ring buffer nằm ở đó
    - feature extraction buffer nằm ở đó
    - tensors nằm ở đó
- Flash lưu model và firmware
- DRAM thường không có nếu là MCU nhỏ

Thực tế triển khai:
- model KWS có thể chỉ vài chục KB tới vài trăm KB Flash
- nhưng vẫn cần SRAM đủ cho feature window + tensor arena

Vấn đề hay gặp:
- model vừa Flash nhưng arena không đủ SRAM
- hoặc audio pipeline ăn SRAM quá nhiều nên phải giảm frame size, giảm window, giảm số mel bins

### 10.2 Vibration anomaly detection trên máy móc

Thiết bị gắn vào motor/cánh quạt:

- Flash:
    - firmware
    - model anomaly detection
    - threshold

- SRAM:
    - sensor buffer của accelerometer
    - FFT buffer
    - feature vector
    - activations

- Flash hoặc FRAM/eMMC:
    - log sự kiện bất thường

Ở đây model thường nhỏ, nhưng phần DSP có thể ngốn SRAM:

- ring buffer
- FFT intermediate
- overlap windows

Nhiều khi DSP còn tốn RAM hơn model.

### 10.3 Person detection bằng camera trên MCU

Đây là bài toán khó hơn wake word nhiều.

Ví dụ ảnh grayscale 96x96 đã là khá nặng với MCU nhỏ.

Nếu dùng RGB, buffer còn lớn hơn nhiều.

Bộ nhớ cần cho:
- frame buffer camera
- pre-processing
- input tensor
- intermediate feature maps
- output tensor

Ở đây:
- SRAM là vấn đề số 1
- Flash là vấn đề số 2

Một board có thể có đủ Flash chứa model 500 KB, nhưng không đủ SRAM để chứa:
- ảnh
- activations
- camera DMA buffer

Nên person detection trên MCU thường phải:
- giảm resolution
- dùng grayscale
- quantize int8
- giảm số channel
- dùng depthwise separable conv
- stream theo tile nếu framework hỗ trợ

## 11) Ứng dụng thực tế trong Edge AI

### 11.1 Smart camera chạy Linux + NPU

Ví dụ camera AI nhận 1080p:
- OS + app trong eMMC/UFS/SSD -> bản chất là Flash storage
- khi app chạy, model được load lên DRAM
- frame từ camera vào DRAM
- NPU lấy dữ liệu từ DRAM, đôi khi copy sang local SRAM
- intermediate tiles chạy trong local SRAM/cache
- kết quả về DRAM rồi app xử lý tiếp

Ở hệ thống này:
- Flash = nơi lưu lâu dài model và software
- DRAM = nơi chạy thật
- SRAM = nơi compute engine tăng tốc thật sự

### 11.2 Smartphone on-device AI

Trên điện thoại:
- model nằm trong storage flash (UFS/NAND based)
- runtime load model vào DRAM
- CPU/GPU/NPU dùng cache SRAM, shared SRAM, system cache
- activations chạy chủ yếu trong DRAM + on-chip SRAM/cache

Điểm quan trọng:
- dù điện thoại có nhiều GB DRAM, hiệu năng AI vẫn bị ràng buộc bởi việc di chuyển dữ liệu giữa DRAM và compute engine
- nên kiến trúc accelerator hiện đại cực kỳ quan tâm tới local SRAM

## 12) Tại sao trong AI, “data movement” quan trọng hơn nhiều người tưởng?

Trong edge AI, không phải phép nhân cộng luôn là thứ đắt nhất.

Nhiều lúc di chuyển dữ liệu mới là thứ tốn năng lượng và thời gian.

Ví dụ:
- lấy weights từ Flash/DRAM vào compute unit
- lấy activation từ DRAM sang cache/SRAM
- ghi output lại DRAM

Nếu mỗi phép toán đều phải chờ bộ nhớ chậm, compute sẽ bị đói dữ liệu.

Đây là lý do:
- SRAM gần compute rất giá trị
- compiler/runtime cố gắng tile data
- kiến trúc NPU thường quảng cáo local SRAM lớn
- quantization giúp giảm traffic memory chứ không chỉ giảm model size

## 13) Quantization ảnh hưởng thế nào tới SRAM, DRAM, Flash?

Đây là chỗ liên hệ trực tiếp với TinyML/Edge AI.

### 13.1 Giảm Flash footprint

**Ví dụ**:
- weight float32: 4 byte/weight
- int8: 1 byte/weight

Nếu quantize từ float32 sang int8:
- kích thước model có thể giảm khoảng 4 lần

Điều này giúp:
- fit vào Flash dễ hơn
- load từ storage sang RAM nhanh hơn
- giảm memory bandwidth

### 13.2 Giảm RAM cho activations

Nếu activations cũng là int8 thay vì float32:
- SRAM/DRAM cho tensor trung gian cũng giảm mạnh

Trong TinyML, đây có thể là khác biệt giữa:
- chạy được
- và không chạy được

### 13.3 Giảm energy và tăng speed

Vì ít byte hơn:
- đọc từ Flash/DRAM ít hơn
- cache/SRAM chứa được nhiều hơn
- throughput tốt hơn

Nhưng không phải mọi backend đều hưởng lợi như nhau:
- MCU/DSP/NPU int8 thường hưởng lợi rất rõ
- một số CPU general-purpose thì speedup phụ thuộc implementation

## 14) Một lỗi tư duy rất hay gặp: “Flash nhiều thì chạy model lớn được”

Không hẳn.

Ví dụ:
- Chip A: Flash 16 MB, SRAM 320 KB
- Chip B: Flash 4 MB, SRAM 1.5 MB

Với một số model vision/audio:
- Chip A có thể chứa model lớn hơn
- nhưng Chip B mới là chip chạy inference mượt hơn, vì activations cần SRAM

Nói cách khác:
- Flash quyết định bạn lưu được gì
- SRAM/DRAM quyết định bạn chạy được gì

Trong TinyML, câu thứ hai thường quan trọng hơn.

## 15) Khi nào Flash là bottleneck, khi nào SRAM là bottleneck?

Flash bottleneck khi:
- model quá lớn
- firmware + model + assets không fit
- cần chứa nhiều model
- OTA update cần thêm vùng dự phòng
- log/config/voice packs chiếm nhiều chỗ

SRAM bottleneck khi:
- tensor arena quá lớn
- audio/camera buffer quá lớn
- activations peak quá cao
- pipeline có nhiều stage cùng tồn tại
- RTOS/task stack ăn RAM
- dùng float thay vì int8

Trong TinyML thực tế, **SRAM bottleneck xảy ra rất thường xuyên**.

## 16) DRAM bottleneck trong Edge AI là gì?

Với edge device lớn hơn, vấn đề không còn chỉ là “đủ hay không đủ”, mà là:
- bandwidth DRAM có đủ không
- latency DRAM có kéo FPS xuống không
- nhiều tác vụ cùng tranh chấp DRAM không
- copy giữa CPU/GPU/NPU có quá nhiều không

Ví dụ:
- camera stream 4K
- ISP, display, video encode, NPU inference cùng chạy
- tất cả tranh DRAM bandwidth

Khi đó dù NPU mạnh, hiệu năng thực vẫn có thể không như lý thuyết.

## 17) Phân biệt vai trò trong một thiết bị TinyML điển hình

Lấy ví dụ một thiết bị nghe wake word:

- Flash chứa:
    - bootloader
    - firmware app
    - model int8
    - label map
    - threshold config

- SRAM chứa
    - audio DMA buffer
    - ring buffer 1–2 giây
    - MFCC/mel features
    - tensor arena
    - stack/heap
    - kết quả suy luận tạm thời

- Hành vi lúc runtime
    - bật nguồn -> code đọc từ Flash
    - microphone stream vào SRAM
    - model inference dùng SRAM làm vùng làm việc
    - nếu detect -> ghi event nhỏ vào Flash hoặc gửi BLE/Wi-Fi

Cái này phản ánh đúng bản chất:
- Flash = lâu dài
- SRAM = nơi làm việc tức thời

## 18) Phân biệt vai trò trong một thiết bị Edge AI camera

Ví dụ smart doorbell có nhận diện người:

### Storage flash

- Linux image
- app
- model file
- logs
- snapshots

### DRAM

- OS memory
- video buffers
- decoded frames
- inference tensors
- user-space app buffers

### SRAM/cache/local memory

- CPU caches
- NPU on-chip SRAM
- DMA staging buffers
- tile buffers

Ở đây:
- Flash không trực tiếp là nơi inference “ngốn” nhiều nhất
- DRAM là nơi chạy chính
- SRAM local là thứ quyết định accelerator hiệu quả ra sa

## Khi chọn phần cứng cho TinyML, nên nhìn gì?

Nếu mục tiêu là TinyML/Edge AI, thay vì chỉ nhìn “có bao nhiêu MB”, nên nhìn tách riêng:

### Với MCU TinyML

- Flash bao nhiêu?
- SRAM bao nhiêu?
- Có external PSRAM không?
- Có DSP/NPU không?
- Có camera/audio peripheral phù hợp không?
- Có hỗ trợ int8 tốt không?
- Framework hỗ trợ mức nào?

Thực tế:
- cho keyword spotting nhỏ, Flash và SRAM vừa phải có thể đủ
- cho vision, SRAM tăng lên rất quan trọng
- external PSRAM có thể cứu được vài use case, nhưng không nhanh như on-chip SRAM

### Với Edge AI device

- DRAM capacity
- DRAM bandwidth
- storage speed
- local SRAM/cache/NPU SRAM
- zero-copy pipeline support
- memory sharing giữa ISP, NPU, GPU, CPU

## Một vài ví dụ quyết định thiết kế thực tế

### Trường hợp A: Wake word đơn giản

Nhu cầu:
- model nhỏ
- audio streaming liên tục
- luôn bật
- tiết kiệm điện

Ưu tiên:
- đủ SRAM cho audio + arena
- Flash đủ cho model + firmware
- không cần DRAM lớn

Đây là sân của MCU TinyML.

### Trường hợp B: Camera nhận diện người trên pin

Nhu cầu:
- frame buffer lớn
- CNN activations lớn
- pre/post-processing nặng hơn

Ưu tiên:
- SRAM rất lớn hoặc có external RAM
- model int8 gọn
- accelerator hỗ trợ vision

### Trường hợp C: Smart camera nhiều model

Nhu cầu:
- detection + tracking + OCR
- Linux stack
- camera pipeline phức tạp

Ưu tiên:
- DRAM lớn
- storage flash đủ
- NPU có local SRAM mạnh
- memory bandwidth tốt